In [1]:
import pandas as pd
import numpy as np

In [8]:
filename = '../data/vindr-full/data/finding_annotations.csv'
df = pd.read_csv(filename)
print(df.shape)
df.columns

(20486, 16)


Index(['study_id', 'series_id', 'image_id', 'laterality', 'view_position',
       'height', 'width', 'breast_birads', 'breast_density',
       'finding_categories', 'finding_birads', 'xmin', 'ymin', 'xmax', 'ymax',
       'split'],
      dtype='str')

In [19]:
# Detect rows with 'No Finding' (handles string lists like "['No Finding']" or "['Nipple Retraction', 'Mass']")
df['has_no_finding'] = df['finding_categories'].astype(str).str.contains(r'No Finding', case=False, na=False)

# Find study_ids where EVERY row for that study_id has 'No Finding'
study_ids_no_finding = (
    df.groupby('study_id')['has_no_finding']
    .agg(all_have_no_finding='all')
    .query('all_have_no_finding')
    .index
    .tolist()
)

print(f"Found {len(study_ids_no_finding)} study_ids where ALL rows are 'No Finding'.")

# Filter the full original rows for these study_ids only
df_no_finding_full = df[df['study_id'].isin(study_ids_no_finding)].copy()
df_no_finding_full = df_no_finding_full.drop(columns=['has_no_finding'])

# Save to a new CSV with ALL original columns (full rows)
df_no_finding_full.to_csv('../data/vindr/no_findings.csv', index=False)

print(f"Saved {len(df_no_finding_full)} full rows to 'no_finding_full_rows.csv'")
print(f"These rows belong to the {len(study_ids_no_finding)} normal (no-finding) studies.")

# quick summary
# print("\nSummary of saved data:")
# print(df_no_finding_full['split'].value_counts())
# print(df_no_finding_full[['study_id', 'series_id', 'image_id']].nunique())


Found 4076 study_ids where ALL rows are 'No Finding'.
Saved 16304 full rows to 'no_finding_full_rows.csv'
These rows belong to the 4076 normal (no-finding) studies.
